<a href="https://colab.research.google.com/github/beaclavarino/AMD-Market-Basket-Analysis-Project/blob/main/project_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market-Basket Analysis

**Algorithms for Massive Data** — Master in Data Science for Economics

---

Beatrice Clavarino — beatrice.clavarino@studenti.unimi.it

*March 2026*

## **1.Dataset**

This project implements a system for discovering frequent itemsets through Market-Basket Analysis, applied to the IMDB Top 1000 Movies dataset.
Following the market-basket model, each movie is treated as a basket, and the actors listed under the Star1, Star2, Star3, and Star4 fields are treated as items. The goal is to identify sets of actors that appear together frequently across movies, that is, itemsets whose support meets or exceeds a minimum threshold *s*.\
The algorithms implemented follow designs that are intended to scale to datasets exceeding available main memory. While the dataset used in this project is small enough to fit in memory, both A-Priori and SON are structured to minimize main-memory usage: A-Priori by counting only pairs of frequent items, and SON by processing data in partitions and making two passes over the full dataset.


In [ ]:
import os
import itertools
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, desc, explode, size,
    array, array_remove, array_distinct,
    lower, trim, monotonically_increasing_id
)
import pyspark.sql.functions as F

In [ ]:
os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
!unzip -o imdb-dataset-of-top-1000-movies-and-tv-shows.zip

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
  0% 0.00/175k [00:00<?, ?B/s]
100% 175k/175k [00:00<00:00, 310MB/s]
Archive:  imdb-dataset-of-top-1000-movies-and-tv-shows.zip
  inflating: imdb_top_1000.csv       


## **2.Data Organization**

### **2.1 Spark**


To handle the computational demands of large-scale data processing, Apache Spark is used as the distributed computing framework. The session is configured with 64 shuffle partitions to ensure efficient parallelism.

In [ ]:
spark = (
    SparkSession.builder
    .appName("MarketBasketAnalysis")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

### **2.2 Data Discovery**

This phase involves initializing the computing environment and exploring the raw dataset to map its structure onto the components of the market-basket model.

In [ ]:
# Data discovery
df_original = spark.read.csv('imdb_top_1000.csv', header=True, inferSchema=True)
df_original.show(5)
df_original.printSchema()
total_rows = df_original.count()
print(f"Total rows: {total_rows}")

+--------------------+--------------------+-------------+-----------+-------+--------------------+-----------+--------------------+----------+--------------------+--------------+--------------+-------------+--------------+-----------+-----------+
|         Poster_Link|        Series_Title|Released_Year|Certificate|Runtime|               Genre|IMDB_Rating|            Overview|Meta_score|            Director|         Star1|         Star2|        Star3|         Star4|No_of_Votes|      Gross|
+--------------------+--------------------+-------------+-----------+-------+--------------------+-----------+--------------------+----------+--------------------+--------------+--------------+-------------+--------------+-----------+-----------+
|https://m.media-a...|The Shawshank Red...|         1994|          A|142 min|               Drama|        9.3|Two imprisoned me...|        80|      Frank Darabont|   Tim Robbins|Morgan Freeman|   Bob Gunton|William Sadler|    2343110| 28,341,469|
|https://m.m

The raw dataset contains 1,000 records and 16 columns, of which only 5 are relevant to this analysis: Series_Title (the basket identifier) and Star1, Star2, Star3, Star4 (the items). The remaining 11 columns are discarded in the preprocessing phase.


### **2.3 Basket Identifier Validation**

A preliminary audit is performed to identify any missing or null values within the Series_Title column.

In [ ]:
# Title missing values check
missing_titles = df_original.filter(
    col("Series_Title").isNull() |
    (col("Series_Title") == "")
).count()

print(f"Missing titles: {missing_titles}")

Missing titles: 0


In order to guarantee the integrity of the model, the uniqueness of the titles is verified.
To ensure a rigorous discovery phase, movie titles are first normalized.

In [ ]:
# Title uniqueness check
df_check = df_original.withColumn("Normalized_Title", trim(lower(col("Series_Title"))))
total_rows = df_original.count()
unique_normalized = df_check.select("Normalized_Title").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Unique normalized titles: {unique_normalized}")

Total rows: 1000
Unique normalized titles: 999


A duplicate Series_Title has been found. It is necessary to determine whether this was an input error or a case of homonymy between different works.

In [ ]:
# Check for movies with the same title
duplicates_check = df_check.groupBy("Normalized_Title").count().filter(F.col("count") > 1)
df_check.join(duplicates_check, "Normalized_Title") \
             .select("Normalized_Title", "Star1", "Star2", "Released_Year") \
             .show()

+----------------+----------+------------+-------------+
|Normalized_Title|     Star1|       Star2|Released_Year|
+----------------+----------+------------+-------------+
|        drishyam|  Mohanlal|       Meena|         2013|
|        drishyam|Ajay Devgn|Shriya Saran|         2015|
+----------------+----------+------------+-------------+



There are two different films with the same title.\
To ensure the integrity of the Market-Basket model and prevent data collision, which would incorrectly merge two different casts into a single basket, the title cannot be used as a primary key. \
In the next steps, a unique ID will be created.

## **3.Preprocessing**

### **3.1 Dimensionality Reduction**

The dataset is restricted to the five relevant columns (Series_Title, Star1, Star2, Star3, Star4), reducing the data footprint by eliminating 11 of the original 16 columns. This ensures that main memory is reserved for the computationally intensive step of counting item and pair occurrences.

In [ ]:
# Selection of useful columns
useful_cols = ["Series_Title", "Star1", "Star2", "Star3", "Star4"]
df = df_original.select(useful_cols)
df.show(5)

+--------------------+--------------+--------------+-------------+--------------+
|        Series_Title|         Star1|         Star2|        Star3|         Star4|
+--------------------+--------------+--------------+-------------+--------------+
|The Shawshank Red...|   Tim Robbins|Morgan Freeman|   Bob Gunton|William Sadler|
|       The Godfather| Marlon Brando|     Al Pacino|   James Caan|  Diane Keaton|
|     The Dark Knight|Christian Bale|  Heath Ledger|Aaron Eckhart| Michael Caine|
|The Godfather: Pa...|     Al Pacino|Robert De Niro|Robert Duvall|  Diane Keaton|
|        12 Angry Men|   Henry Fonda|   Lee J. Cobb|Martin Balsam|  John Fiedler|
+--------------------+--------------+--------------+-------------+--------------+
only showing top 5 rows


### **3.2 Unique IDs**

To resolve the duplicate title issue identified during data discovery, a unique Basket_ID is generated for each row using monotonically_increasing_id(). This guarantees that each movie is treated as a distinct basket, regardless of title.

In [ ]:
# Unique IDs
df = df.withColumn("Basket_ID", monotonically_increasing_id())
cols = ["Basket_ID"] + [c for c in df.columns if c != "Basket_ID"]
df = df.select(cols)
df.show(5)

+---------+--------------------+--------------+--------------+-------------+--------------+
|Basket_ID|        Series_Title|         Star1|         Star2|        Star3|         Star4|
+---------+--------------------+--------------+--------------+-------------+--------------+
|        0|The Shawshank Red...|   Tim Robbins|Morgan Freeman|   Bob Gunton|William Sadler|
|        1|       The Godfather| Marlon Brando|     Al Pacino|   James Caan|  Diane Keaton|
|        2|     The Dark Knight|Christian Bale|  Heath Ledger|Aaron Eckhart| Michael Caine|
|        3|The Godfather: Pa...|     Al Pacino|Robert De Niro|Robert Duvall|  Diane Keaton|
|        4|        12 Angry Men|   Henry Fonda|   Lee J. Cobb|Martin Balsam|  John Fiedler|
+---------+--------------------+--------------+--------------+-------------+--------------+
only showing top 5 rows


### **3.3 Text Normalization**

To ensure accurate support counts, actor names are standardized through:

- Case folding (conversion to lowercase) to eliminate case inconsistencies that would otherwise produce false negatives (e.g., treating "Al Pacino" and "al pacino" as different items).
- Trimming (removal of leading/trailing whitespace) to handle formatting artifacts.
- Null imputation: missing values in actor fields are replaced with empty strings, to be filtered out in the next step.

In [ ]:
# Text Normalization
for c in ["Star1", "Star2", "Star3", "Star4"]:
    df = df.withColumn(c, trim(lower(col(c))))
    df = df.fillna({c: ""})
df.show(5)

+---------+--------------------+--------------+--------------+-------------+--------------+
|Basket_ID|        Series_Title|         Star1|         Star2|        Star3|         Star4|
+---------+--------------------+--------------+--------------+-------------+--------------+
|        0|The Shawshank Red...|   tim robbins|morgan freeman|   bob gunton|william sadler|
|        1|       The Godfather| marlon brando|     al pacino|   james caan|  diane keaton|
|        2|     The Dark Knight|christian bale|  heath ledger|aaron eckhart| michael caine|
|        3|The Godfather: Pa...|     al pacino|robert de niro|robert duvall|  diane keaton|
|        4|        12 Angry Men|   henry fonda|   lee j. cobb|martin balsam|  john fiedler|
+---------+--------------------+--------------+--------------+-------------+--------------+
only showing top 5 rows


### **3.4 Basket Transformation and Integrity Validation**

Each row is transformed into a transactional format, where the four actor columns are aggregated into a single array using Spark's array() function. Empty strings introduced during null imputation are then removed with array_remove(), ensuring each basket contains only valid items.

In [ ]:
# Basket Trasformation
df = df.select(
    col("Basket_ID"),
    col("Series_Title"),
    array_remove(
        array(col("Star1"), col("Star2"), col("Star3"), col("Star4")),
        ""
    ).alias("Actors")
)

df.show(5)

+---------+--------------------+--------------------+
|Basket_ID|        Series_Title|              Actors|
+---------+--------------------+--------------------+
|        0|The Shawshank Red...|[tim robbins, mor...|
|        1|       The Godfather|[marlon brando, a...|
|        2|     The Dark Knight|[christian bale, ...|
|        3|The Godfather: Pa...|[al pacino, rober...|
|        4|        12 Angry Men|[henry fonda, lee...|
+---------+--------------------+--------------------+
only showing top 5 rows


Two final constraints are applied to comply with the formal definition of a basket:

- array_distinct() ensures each actor appears at most once per basket, preventing duplicate counts within a single transaction.
- Empty basket filtering removes any movie with no listed actors, as these contribute no information to the analysis.

In [ ]:
# Integrity Validation
df = df.withColumn("Actors", array_distinct(col("Actors")))
df = df.filter(size(col("Actors")) > 0)
df.show(5)

+---------+--------------------+--------------------+
|Basket_ID|        Series_Title|              Actors|
+---------+--------------------+--------------------+
|        0|The Shawshank Red...|[tim robbins, mor...|
|        1|       The Godfather|[marlon brando, a...|
|        2|     The Dark Knight|[christian bale, ...|
|        3|The Godfather: Pa...|[al pacino, rober...|
|        4|        12 Angry Men|[henry fonda, lee...|
+---------+--------------------+--------------------+
only showing top 5 rows


In [ ]:
df.show(5)
df.printSchema()
total_rows = df.count()
print(f"Total rows: {total_rows}")

+---------+--------------------+--------------------+
|Basket_ID|        Series_Title|              Actors|
+---------+--------------------+--------------------+
|        0|The Shawshank Red...|[tim robbins, mor...|
|        1|       The Godfather|[marlon brando, a...|
|        2|     The Dark Knight|[christian bale, ...|
|        3|The Godfather: Pa...|[al pacino, rober...|
|        4|        12 Angry Men|[henry fonda, lee...|
+---------+--------------------+--------------------+
only showing top 5 rows
root
 |-- Basket_ID: long (nullable = false)
 |-- Series_Title: string (nullable = true)
 |-- Actors: array (nullable = false)
 |    |-- element: string (containsNull = false)

Total rows: 1000


## **4.Algorithms**

### **4.1 Support Threshold ($s$)**

In [ ]:
# Total number of baskets (N)
N = df.count()

# Support threshold (s)
support_fraction = 0.005
s = N * support_fraction

print(f"Total Baskets: {N}")
print(f"Support Threshold (s): {int(s)}")

Total Baskets: 1000
Support Threshold (s): 5


An initial threshold of $\text{support_fraction} = 0.005$ was tested, yielding $s = 5$ on the basket dataset.

### **4.2 A-priori Algorithm**

#### **4.2.1 A-Priori:Frequent Singletons ($L_1$)**

In the first pass, each actor is counted individually across all baskets. Only those actors whose count meets or exceeds
$s$ are retained as
frequent singletons, forming the set $L_1$​.

In [ ]:
# Frequent Singletons (L1)
L1 = (
    df
    .select(explode(col("Actors")).alias("Actor"))
    .groupBy("Actor")
    .count()
    .filter(col("count") >= s)
    .sort(desc("count"))
)
print(f"Support Threshold (s): {int(s)}")
print("Number of Frequent Singletons:", L1.count())
L1.show(5)

Support Threshold (s): 5
Number of Frequent Singletons: 78
+--------------+-----+
|         Actor|count|
+--------------+-----+
|robert de niro|   17|
|     tom hanks|   14|
|     al pacino|   13|
|     brad pitt|   12|
|clint eastwood|   12|
+--------------+-----+
only showing top 5 rows


With $s = 5$, the algorithm identified 78 frequent singletons. The top actors
by frequency are Robert De Niro (17 appearances), Tom Hanks (14), and Al Pacino (13).

By the Monotonicity Property, only items in $L_1$ can be part of any frequent pair.
This reduces the candidate space from all $\binom{n}{2}$ possible pairs to only
pairs drawn from the 78 frequent items.

#### **4.2.2 A-Priori: Frequent Pairs ($L_2$)**

In the second pass, pairs are generated exclusively from items in $L_1$
. For each basket, infrequent actors are filtered out, and all combinations of 2 elements among the remaining frequent actors are generated. Each pair is then counted using a reduceByKey operation.

In [ ]:
# Frequent Pairs (L2)
frequent_actors_set = set(L1.select("Actor").rdd.flatMap(lambda x: x).collect())

def generate_frequent_pairs(actors):
    filtered = [a for a in actors if a in frequent_actors_set]
    if len(filtered) >= 2:
        return [tuple(sorted(p)) for p in itertools.combinations(filtered, 2)]
    return []

L2 = (
    df.rdd
    .flatMap(lambda row: generate_frequent_pairs(row.Actors))
    .map(lambda p: (p, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[1] >= s)
    .map(lambda x: Row(items=list(x[0]), freq=x[1]))
    .toDF()
)
print(f"Support Threshold (s): {int(s)}")
print("Number of Frequent Pairs:", L2.count())
L2.show(5)

Support Threshold (s): 5
Number of Frequent Pairs: 3
+--------------------+----+
|               items|freq|
+--------------------+----+
|[daniel radcliffe...|   5|
|[daniel radcliffe...|   6|
|[emma watson, rup...|   5|
+--------------------+----+



With $s=5$, only
3 frequent pairs were found. This result highlights the high sparsity of the dataset: with a maximum of 4 actors per basket, the number of co-occurrences is inherently limited.

#### **4.2.3 Revised Support Threshold**

A support of 0.5% proved too restrictive for this dataset. To discover a broader range of actor collaborations, the threshold was lowered to $\text{support_fraction} = 0.003$, yielding $s=3$.

This decision reflects a key practical consideration: the support threshold must be calibrated to the structure of the data. A threshold that is too high yields too few results, while one that is too low produces too many candidates for main memory to handle efficiently.

In [ ]:
# Revised support threshold for 0.3% support
new_support_fraction = 0.003
new_s = N * new_support_fraction

print(f"Total Baskets: {N}")
print(f"New Support Threshold (s): {int(new_s)}")

Total Baskets: 1000
New Support Threshold (s): 3


In [ ]:
# Revised Frequent Singletons (L1)
new_L1 = (
    df
    .select(explode(col("Actors")).alias("Actor"))
    .groupBy("Actor")
    .count()
    .filter(col("count") >= new_s)
    .sort(desc("count"))
)

print(f"Support Threshold (s): {int(new_s)}")
print("Number of Frequent Singletons (L1):", new_L1.count())
new_L1.show(5)

Support Threshold (s): 3
Number of Frequent Singletons (L1): 272
+--------------+-----+
|         Actor|count|
+--------------+-----+
|robert de niro|   17|
|     tom hanks|   14|
|     al pacino|   13|
|     brad pitt|   12|
|clint eastwood|   12|
+--------------+-----+
only showing top 5 rows


In [ ]:
# Revised Frequent Pairs (L2)
frequent_actors_set = set(new_L1.select("Actor").rdd.flatMap(lambda x: x).collect())

new_L2 = (
    df.rdd
    .flatMap(lambda row: generate_frequent_pairs(row.Actors))
    .map(lambda p: (p, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[1] >= new_s)
    .map(lambda x: Row(items=list(x[0]), freq=x[1]))
    .toDF()
)

print(f"Support Threshold (s): {int(new_s)}")
print("Number of Frequent Pairs (L2):", new_L2.count())
new_L2.show(5)

Support Threshold (s): 3
Number of Frequent Pairs (L2): 25
+--------------------+----+
|               items|freq|
+--------------------+----+
|[al pacino, diane...|   3|
|[christian bale, ...|   3|
|[al pacino, rober...|   3|
|[elijah wood, ian...|   3|
|[elijah wood, orl...|   3|
+--------------------+----+
only showing top 5 rows


With $s = 3$:
- $L_1$ grew to 272 frequent singletons.
- $L_2$ contains 25 frequent pairs.

The A-Priori principle demonstrated its value: by generating pairs only from $L_1$
members, we avoided evaluating $\binom{272}{2} = 36{,}856$ potential candidate pairs.
Only 25 survived the threshold, validating the Monotonicity Property as an effective
tool for managing the quadratic growth of candidates.

#### **4.2.4 A-Priori: Association Rules**

In [ ]:
singleton_support = {
    row['Actor']: row['count']
    for row in new_L1.collect()
}

pair_support = {
    tuple(sorted(row['items'])): row['freq']
    for row in new_L2.collect()
}

In [ ]:
actor_movies = (
    df.select(col("Series_Title"), explode(col("Actors")).alias("Actor"))
)

pair_with_movies = (
    actor_movies.alias("a")
    .join(actor_movies.alias("b"), on="Series_Title")
    .filter(col("a.Actor") < col("b.Actor"))
    .select(
        col("a.Actor").alias("actor_A"),
        col("b.Actor").alias("actor_B"),
        col("Series_Title").alias("movie")
    )
    .groupBy("actor_A", "actor_B")
    .agg(F.collect_set("movie").alias("movies"))
)

In [ ]:
rules = []
for pair, freq in pair_support.items():
    for i in range(len(pair)):
        antecedent = pair[i]
        consequent = pair[1 - i]

        s_ant = singleton_support.get(antecedent)
        s_cons = singleton_support.get(consequent)

        if s_ant and s_cons:
            confidence = freq / s_ant
            prob_consequent = s_cons / N
            interest = confidence - prob_consequent

            rules.append(Row(
                antecedent=[antecedent],
                consequent=[consequent],
                confidence=float(confidence),
                interest=float(interest)
            ))

apriori_rules = spark.createDataFrame(rules)


In [ ]:
apriori_rules_with_movies = (
    apriori_rules
    .withColumn("antecedent_str", col("antecedent")[0])
    .withColumn("consequent_str", col("consequent")[0])
    .join(
        pair_with_movies,
        (
            (col("antecedent_str") == col("actor_A")) &
            (col("consequent_str") == col("actor_B"))
        ) | (
            (col("antecedent_str") == col("actor_B")) &
            (col("consequent_str") == col("actor_A"))
        ),
        how="left"
    )
    .drop("antecedent_str", "consequent_str", "actor_A", "actor_B")
)

apriori_rules_with_movies.sort(F.desc("interest")).show(20, truncate=False)

+------------------+-------------------+------------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|antecedent        |consequent         |confidence        |interest          |movies                                                                                                                                                                                                                                                    |
+------------------+-------------------+------------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[elijah w


The top 20 rules sorted by Interest reveal strong collaborative patterns, all with Confidence = 1.0 and Interest > 0.98.

High Interest values close to 1 indicate that these actors are rare in the global dataset (low $\Pr[B]$), but appear almost exclusively together — a strong positive correlation that is not a statistical artifact of high individual frequency.

Since A-Priori in this implementation generates rules exclusively from frequent pairs ($L_2$), the antecedent and consequent are always single actors. Rules involving larger itemsets are explored in the SON algorithm section.

### **4.3 SON Algorithm**

#### **4.3.1 SON: Local Candidates Generation**


The dataset is divided into $k = 2$ partitions, a reasonable choice given the small
size of the dataset (1,000 baskets). Each partition represents a fraction $p = 0.5$
of the total data. The local support threshold is therefore:

$$local\_s = p \times s = 0.5 \times 3 = 1.5$$

As a result, only candidates appearing at least 2 times within a partition are
retained as local frequent itemsets.

In [ ]:
#Local Candidates Generation
def find_local_frequent_scalable(partition_data, support_fraction):
    local_baskets = [set(b.Actors) for b in partition_data]
    if not local_baskets:
        return []

    local_s = len(local_baskets) * support_fraction
    all_frequent = []

    item_counts = {}
    for b in local_baskets:
        for item in b:
            item_counts[item] = item_counts.get(item, 0) + 1

    current_L = {
        tuple(sorted([item])): count
        for item, count in item_counts.items()
        if count >= local_s
    }
    all_frequent.extend(current_L.keys())

    singleton_set = set(current_L.keys())
    current_L_set = set(current_L.keys())

    k = 2
    while current_L:
        candidate_counts = {}

        for b in local_baskets:
            filtered_basket = sorted([
                item for item in b if (item,) in singleton_set
            ])

            if len(filtered_basket) >= k:
                for cand in itertools.combinations(filtered_basket, k):
                    sub_sets = list(itertools.combinations(cand, k - 1))
                    if all(s in current_L_set for s in sub_sets):
                        candidate_counts[cand] = candidate_counts.get(cand, 0) + 1

        current_L = {
            cand: count
            for cand, count in candidate_counts.items()
            if count >= local_s
        }

        if current_L:
            all_frequent.extend(current_L.keys())
            current_L_set = set(current_L.keys())
            k += 1
        else:
            break

    return [i[0] if len(i) == 1 else i for i in all_frequent]


The candidates from all partitions are unified and counted against the full dataset.

In [ ]:
# Candidate Union from all Partitions
N = df.count()
support_fraction = 0.003
s = N * support_fraction

global_candidates = (
    df.repartition(2).rdd
    .mapPartitions(lambda p: find_local_frequent_scalable(p, support_fraction))
    .distinct()
    .collect()
)

print(f"Total global candidates: {len(global_candidates)}")

Total global candidates: 550



The first pass produced 550 global candidates across all itemset sizes.

#### **4.3.2 SON: Global Candidates Validation**

Any candidate whose global support is below $s$ is discarded as a false positive.


In [ ]:
# Final Counting and Filtering
def count_all_final(basket, candidates):
    found = []
    actors_set = set(basket.Actors)
    for c in candidates:
        if isinstance(c, str):
            if c in actors_set:
                found.append((c, 1))
        elif all(item in actors_set for item in c):
            found.append((c, 1))
    return found

final_itemsets_rdd = (
    df.rdd
    .flatMap(lambda b: count_all_final(b, global_candidates))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[1] >= s)
)

The last section of the code transforms the raw results into a tabular structure and displays them according to their cardinality.

In [ ]:
# DataFrame Transformation for Streamlined Analysis
df_final_son = final_itemsets_rdd.map(
    lambda x: Row(
        items=list(x[0]) if not isinstance(x[0], str) else [x[0]],
        size=len(x[0]) if not isinstance(x[0], str) else 1,
        freq=x[1]
    )
).toDF()

# Extraction of the Maximum Cardinality (k) for Result Iteration
max_k = df_final_son.agg({"size": "max"}).collect()[0][0]

if max_k is not None:
    for k in range(1, max_k + 1):
        print(f"\nFrequent itemsets of size {k} (L{k})")
        res_k = df_final_son.filter(col("size") == k)
        print(f"Total found: {res_k.count()}")
        res_k.sort(desc("freq")).show(5, truncate=False)


Frequent itemsets of size 1 (L1)
Total found: 272
+----------------+----+----+
|items           |size|freq|
+----------------+----+----+
|[robert de niro]|1   |17  |
|[tom hanks]     |1   |14  |
|[al pacino]     |1   |13  |
|[brad pitt]     |1   |12  |
|[clint eastwood]|1   |12  |
+----------------+----+----+
only showing top 5 rows

Frequent itemsets of size 2 (L2)
Total found: 25
+--------------------------------+----+----+
|items                           |size|freq|
+--------------------------------+----+----+
|[daniel radcliffe, rupert grint]|2   |6   |
|[daniel radcliffe, emma watson] |2   |5   |
|[emma watson, rupert grint]     |2   |5   |
|[joe pesci, robert de niro]     |2   |4   |
|[tim allen, tom hanks]          |2   |4   |
+--------------------------------+----+----+
only showing top 5 rows

Frequent itemsets of size 3 (L3)
Total found: 3
+---------------------------------------------+----+----+
|items                                        |size|freq|
+-------------------


The consistency between $L_1$ and $L_2$ across both algorithms confirms the correctness of both implementations.

#### **4.3.3 SON: Association Rules**

In [ ]:
# Association Rules
support_dict = {tuple(sorted(row['items'])): row['freq'] for row in df_final_son.collect()}

def generate_association_rules_theory(df_frequent, support_dict, N):
    rules = []
    frequent_sets = df_frequent.filter(F.col("size") > 1).collect()

    for row in frequent_sets:
        itemset = tuple(sorted(row['items']))
        support_I = row['freq']

        for i in range(1, len(itemset)):
            for antecedent in itertools.combinations(itemset, i):
                consequent = tuple(sorted(set(itemset) - set(antecedent)))

                s_ant = support_dict.get(antecedent)
                s_cons = support_dict.get(consequent)

                if s_ant and s_cons:
                    confidence = support_I / s_ant
                    prob_consequent = s_cons / N
                    interest = confidence - prob_consequent

                    rules.append(Row(
                        antecedent=list(antecedent),
                        consequent=list(consequent),
                        confidence=float(confidence),
                        interest=float(interest)
                    ))
    return rules

rules_list = generate_association_rules_theory(df_final_son, support_dict, N)

if rules_list:
    association_rules = spark.createDataFrame(rules_list)

    # Build movie lookup for any itemset size (works for pairs and triples)
    actor_movies = (
        df.select(col("Series_Title"), explode(col("Actors")).alias("Actor"))
    )

    def get_movies_for_itemset(actors_list):
        # Find movies containing ALL actors in the itemset
        result = actor_movies.filter(col("Actor") == actors_list[0])
        for actor in actors_list[1:]:
            movies_with_actor = actor_movies.filter(col("Actor") == actor).select("Series_Title")
            result = result.join(movies_with_actor, on="Series_Title")
        return result.select("Series_Title").distinct()

    # Add movies column to rules
    def enrich_rules_with_movies(rules_df):
        rules_collected = rules_df.collect()
        enriched = []
        for row in rules_collected:
            full_itemset = list(row['antecedent']) + list(row['consequent'])
            movies = get_movies_for_itemset(full_itemset).rdd.flatMap(lambda x: x).collect()
            enriched.append(Row(
                antecedent=row['antecedent'],
                consequent=row['consequent'],
                confidence=row['confidence'],
                interest=row['interest'],
                movies=movies
            ))
        return spark.createDataFrame(enriched)

    association_rules_with_movies = enrich_rules_with_movies(association_rules)
    association_rules_with_movies.sort(F.desc("interest")).show(20, truncate=False)

+-------------------------------+------------------------------+----------+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|antecedent                     |consequent                    |confidence|interest|movies                                                                                                                                                                                                                                                    |
+-------------------------------+------------------------------+----------+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


Because SON identifies itemsets up to size $L_3$, it produces richer rules than A-Priori, including hierarchical rules derived from triples (e.g., $\{A, B\} \rightarrow C$) in addition to pairwise rules.

The top 20 rules by Interest include:

- Symmetric rules: pairs like $[A] \rightarrow [B]$ and $[B] \rightarrow [A]$ with Confidence = 1.0, indicating perfect co-occurrence.
- Hierarchical rules: e.g., $[\text{Elijah Wood}] \rightarrow [\text{Ian McKellen, Orlando Bloom}]$ with Confidence = 1.0, showing that the presence of one actor in a trilogy perfectly predicts the other two.

All top-20 rules show Interest > 0.99, confirming that these associations are not accidental. The consequent items have low baseline probabilities $\Pr[j]$, yet appear almost exclusively in the presence of the antecedent, a pattern that reflects real-world collaboration structures in film franchises.

### **4.4 Comparison**

The pairwise rules produced by both algorithms are identical, validating the correctness of both implementations. SON additionally extracts rules from $L_3$, which A-Priori (in this implementation) does not reach, as it stops at frequent pairs.

## **5.Scalability Analysis**


To evaluate scalability, the dataset was artificially replicated at 1×, 10×, and 100× its original size, simulating datasets of 1,000, 10,000, and 100,000 baskets respectively. The support threshold is kept as a fixed fraction of $N$ ($\text{support_fraction} = 0.003$), so $s$ scales automatically with dataset size, no parameter tuning is needed as the data grows.

In [ ]:
REPLICATION_FACTORS = [1, 10, 100]

datasets = {}

for factor in REPLICATION_FACTORS:
    df_scaled = df
    for _ in range(factor - 1):
        df_scaled = df_scaled.union(df)
    datasets[factor] = df_scaled
    print(f"Replication {factor}x | Baskets: {df_scaled.count()}")

Replication 1x | Baskets: 1000
Replication 10x | Baskets: 10000
Replication 100x | Baskets: 100000


### **5.1 A-priori Scalability**

In [ ]:
import time

apriori_results = []

for factor in REPLICATION_FACTORS:
    df_scaled = datasets[factor]
    N_scaled = df_scaled.count()
    s_scaled = N_scaled * new_support_fraction

    print(f"A-priori | Replication: {factor}x | Baskets: {N_scaled} | s={int(s_scaled)}")

    # Frequent Singletons (L1)
    new_L1_scaled = (
        df_scaled
        .select(explode(col("Actors")).alias("Actor"))
        .groupBy("Actor")
        .count()
        .filter(col("count") >= s_scaled)
        .sort(desc("count"))
    )
    print(f"Number of Frequent Singletons (L1): {new_L1_scaled.count()}")

    # Frequent Pairs (L2)
    frequent_actors_set = set(new_L1_scaled.select("Actor").rdd.flatMap(lambda x: x).collect())

    start = time.time()

    new_L2_scaled = (
        df_scaled.rdd
        .flatMap(lambda row: generate_frequent_pairs(row.Actors))
        .map(lambda p: (p, 1))
        .reduceByKey(lambda a, b: a + b)
        .filter(lambda x: x[1] >= s_scaled)
        .map(lambda x: Row(items=list(x[0]), freq=x[1]))
        .toDF()
    )

    elapsed = time.time() - start
    n_pairs = new_L2_scaled.count()

    print(f"Number of Frequent Pairs (L2): {n_pairs}")
    print(f"Execution Time: {elapsed:.2f} seconds")

    apriori_results.append((N_scaled, elapsed, n_pairs))

A-priori | Replication: 1x | Baskets: 1000 | s=3
Number of Frequent Singletons (L1): 272
Number of Frequent Pairs (L2): 25
Execution Time: 0.56 seconds
A-priori | Replication: 10x | Baskets: 10000 | s=30
Number of Frequent Singletons (L1): 272
Number of Frequent Pairs (L2): 25
Execution Time: 5.09 seconds
A-priori | Replication: 100x | Baskets: 100000 | s=300
Number of Frequent Singletons (L1): 272
Number of Frequent Pairs (L2): 25
Execution Time: 52.53 seconds


The number of frequent singletons and pairs remains constant across all sizes, as expected: replicating the data does not change the relative frequencies, and $s$ scales proportionally. This confirms that the use of a fractional threshold is the correct design choice for scalability.

Execution time grows approximately sub-linearly with dataset size: a 100× increase in data yields roughly a 85× increase in time, consistent with the linear scan required by A-Priori.

### **5.2 SON Scalability**

In [ ]:
son_results = []

for factor in REPLICATION_FACTORS:
    df_scaled = datasets[factor]
    N_scaled = df_scaled.count()
    s_scaled = N_scaled * new_support_fraction

    print(f"SON | Replication: {factor}x | Baskets: {N_scaled} | s={int(s_scaled)}")

    start = time.time()

    # Local candidates
    global_candidates_scaled = (
        df_scaled.repartition(2).rdd
        .mapPartitions(lambda p: find_local_frequent_scalable(p, new_support_fraction))
        .distinct()
        .collect()
    )
    print(f"Total global candidates: {len(global_candidates_scaled)}")

    # Global counting
    df_final_son_scaled = (
        df_scaled.rdd
        .flatMap(lambda b: count_all_final(b, global_candidates_scaled))
        .reduceByKey(lambda a, b: a + b)
        .filter(lambda x: x[1] >= s_scaled)
        .map(lambda x: Row(
            items=list(x[0]) if not isinstance(x[0], str) else [x[0]],
            size=len(x[0]) if not isinstance(x[0], str) else 1,
            freq=x[1]
        ))
        .toDF()
    )

    elapsed = time.time() - start

    # Results
    max_k = df_final_son_scaled.agg({"size": "max"}).collect()[0][0]
    total_itemsets = 0
    if max_k is not None:
        for k in range(1, max_k + 1):
            res_k = df_final_son_scaled.filter(col("size") == k)
            count_k = res_k.count()
            total_itemsets += count_k
            print(f"  L{k} (size {k}): {count_k} frequent itemsets")

    print(f"Execution Time: {elapsed:.2f} seconds")

    son_results.append((N_scaled, elapsed, total_itemsets))

SON | Replication: 1x | Baskets: 1000 | s=3
Total global candidates: 550
  L1 (size 1): 272 frequent itemsets
  L2 (size 2): 25 frequent itemsets
  L3 (size 3): 3 frequent itemsets
Execution Time: 1.51 seconds
SON | Replication: 10x | Baskets: 10000 | s=30
Total global candidates: 300
  L1 (size 1): 272 frequent itemsets
  L2 (size 2): 25 frequent itemsets
  L3 (size 3): 3 frequent itemsets
Execution Time: 5.27 seconds
SON | Replication: 100x | Baskets: 100000 | s=300
Total global candidates: 300
  L1 (size 1): 272 frequent itemsets
  L2 (size 2): 25 frequent itemsets
  L3 (size 3): 3 frequent itemsets
Execution Time: 66.73 seconds


The frequent itemset counts are again stable across dataset sizes. Notably, the number of global candidates decreases from 550 to 300 after the first replication: with a higher local threshold ($s$ scales with $N$), fewer false positives are generated at the partition level, confirming the theoretical trade-off between partition size and candidate quality.

SON is consistently slower than A-Priori on this dataset, due to the overhead of the two-pass MapReduce structure. However, this cost is justified by the additional capability to detect itemsets of size $k > 2$, and by its suitability for truly distributed environments where data does not fit in main memory.

Both algorithms show approximately linear scaling in execution time with respect to dataset size, demonstrating that they are suitable for deployment on larger real-world datasets.